In [1]:
import numpy as np
import pandas as pd
df = pd.read_csv("C:/Users/inter/OneDrive/Documents/GENAI/Skincare-ChatBot/skincare_ingredients_dataset (2).csv")

In [2]:
df.shape

(2100, 19)

In [3]:
df.isnull().sum()

ingredient_name                 0
category                        0
product_format                  0
primary_function                0
key_benefits                    0
how_to_apply                    0
time_of_use                     0
recommended_frequency           0
typical_concentration        1834
follow_up_with_sunscreen        0
when_to_avoid                   0
best_for_skin_types             0
combines_well_with              0
avoid_combining_with            0
comedogenic_rating_0_to_5       0
typical_ph_range              735
pregnancy_safe                  0
patch_test_recommended          0
price_tier                      0
dtype: int64

In [4]:
df = df.drop('typical_concentration', axis=1)
df = df.drop('typical_ph_range', axis=1)

In [5]:
df.dropna(inplace=True)
df.isnull().sum()

ingredient_name              0
category                     0
product_format               0
primary_function             0
key_benefits                 0
how_to_apply                 0
time_of_use                  0
recommended_frequency        0
follow_up_with_sunscreen     0
when_to_avoid                0
best_for_skin_types          0
combines_well_with           0
avoid_combining_with         0
comedogenic_rating_0_to_5    0
pregnancy_safe               0
patch_test_recommended       0
price_tier                   0
dtype: int64

In [6]:
df_unique = (
    df.sort_values("ingredient_name")
      .drop_duplicates(subset="ingredient_name", keep="first")
      .reset_index(drop=True)
)

In [7]:
print(df.columns)
print(df.head())

Index(['ingredient_name', 'category', 'product_format', 'primary_function',
       'key_benefits', 'how_to_apply', 'time_of_use', 'recommended_frequency',
       'follow_up_with_sunscreen', 'when_to_avoid', 'best_for_skin_types',
       'combines_well_with', 'avoid_combining_with',
       'comedogenic_rating_0_to_5', 'pregnancy_safe', 'patch_test_recommended',
       'price_tier'],
      dtype='str')
                 ingredient_name                               category  \
0                         SNAP-8  Peptide (Neurotransmitter-inhibiting)   
1         Palmitoyl Tripeptide-1           Peptide (Collagen-signaling)   
2           Bakuchi Seed Extract           Botanical (Bakuchiol source)   
3  Vitamin B3 Derivative Complex                    Brightening/Barrier   
4                Borage Seed Oil                 Facial Oil (Botanical)   

   product_format                                   primary_function  \
0      Facial Oil  Extended-action muscle-contraction relaxing pe...   
1

In [8]:
# ...existing code...
cols = [
    "ingredient_name","category","primary_function","key_benefits",
    "recommended_frequency","follow_up_with_sunscreen","best_for_skin_types",
    "combines_well_with","comedogenic_rating_0_to_5","price_tier"
]
df["search_text"] = df[cols].fillna("").astype(str).agg(". ".join, axis=1)
# ...existing code...

In [9]:
df['search_text'].head()

0    SNAP-8. Peptide (Neurotransmitter-inhibiting)....
1    Palmitoyl Tripeptide-1. Peptide (Collagen-sign...
2    Bakuchi Seed Extract. Botanical (Bakuchiol sou...
3    Vitamin B3 Derivative Complex. Brightening/Bar...
4    Borage Seed Oil. Facial Oil (Botanical). High ...
Name: search_text, dtype: str

In [10]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
# Ensure all inputs are strings and keep alignment with the dataframe
texts = df['search_text'].fillna("").astype(str).tolist()
# Encode in batches and return a NumPy array
embeddings = model.encode(texts, show_progress_bar=True, batch_size=64, convert_to_numpy=True)
np.save("Ingredients_embedding.npy", embeddings)
df.to_pickle("ingredient_data.pkl")

c:\Users\inter\OneDrive\Documents\GENAI\Skincare-ChatBot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 33/33 [00:24<00:00,  1.34it/s]


## Vector Database (ChromaDB)

In [14]:
import chromadb
from langchain_chroma import Chroma

# Persistent local vector database — replaces the in-memory NumPy array +
# sklearn cosine_similarity brute-force search below with a proper
# ANN-indexed vector store that persists to disk across notebook runs.
chroma_client = chromadb.PersistentClient(path="./chroma_skincare_db")
collection = chroma_client.get_or_create_collection(
    name="skincare_ingredients",
    metadata={"hnsw:space": "cosine"},  # match the cosine similarity used before
)

# Populate once — skip on reruns so we don't keep re-adding the same rows.
# If the underlying data or embeddings change, delete the
# ./chroma_skincare_db folder (or the collection) to force a rebuild.
if collection.count() == 0:
    collection.add(
        ids=df.index.astype(str).tolist(),
        embeddings=embeddings.tolist(),
        documents=df["search_text"].tolist(),
        metadatas=df[[
            "ingredient_name", "category", "primary_function", "key_benefits",
            "best_for_skin_types", "when_to_avoid", "price_tier",
            "comedogenic_rating_0_to_5", "follow_up_with_sunscreen",
        ]].astype(str).to_dict(orient="records"),
    )

print(f"Collection now holds {collection.count()} ingredients.")


Collection now holds 2100 ingredients.


In [15]:
import sys
print(sys.executable)
%pip show langchain-chroma

c:\Users\inter\OneDrive\Documents\GENAI\Skincare-ChatBot\.venv\Scripts\python.exe
Name: langchain-chroma
Version: 1.1.0
Summary: An integration package connecting Chroma and LangChain.
Home-page: https://docs.langchain.com/oss/python/integrations/providers/chroma
Author: 
Author-email: 
License: MIT
Location: c:\Users\inter\OneDrive\Documents\GENAI\Skincare-ChatBot\.venv\Lib\site-packages
Requires: chromadb, langchain-core, numpy
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [16]:
def retrieve(query, model, collection, df, top_k=5, threshold=0.35, candidate_multiplier=4):
    """Vector-search retrieval via ChromaDB (replaces the previous brute-force
    sklearn cosine_similarity + argsort pass over the full embedding matrix).

    Returns up to top_k * candidate_multiplier rows rather than exactly
    top_k — filter_by_skin_type (applied afterward, in chatbot_reply) needs
    a larger pool to filter down from, matching the original two-stage
    retrieve-then-filter flow.
    """
    n_candidates = min(max(top_k * candidate_multiplier, top_k), collection.count())
    results = collection.query(
        query_embeddings=model.encode([query]).tolist(),
        n_results=n_candidates,
    )

    if not results["ids"][0]:
        return None, 0.0

    similarities = [1 - d for d in results["distances"][0]]  # cosine distance -> similarity
    max_sim = similarities[0]  # Chroma orders results by distance ascending

    if max_sim < threshold:
        return None, max_sim

    indices = [int(i) for i in results["ids"][0]]
    retrieved = df.loc[indices].copy()
    retrieved["similarity"] = similarities

    return retrieved, max_sim


In [17]:
def build_context(retrieved_df):
    lines = []
    for _, row in retrieved_df.iterrows():
        lines.append(
            f"- {row.ingredient_name}: {row.primary_function} "
            f"(benefits: {row.key_benefits}; best for: {row.best_for_skin_types}; "
            f"avoid when: {row.when_to_avoid})"
        )
    return "\n".join(lines)

In [18]:
SKIN_TYPES = ["oily", "dry", "combination", "sensitive", "normal", "mature", "acne-prone"]

def filter_by_skin_type(query, df):
    mentioned = [t for t in SKIN_TYPES if t in query.lower()]
    if not mentioned:
        return df
    mask = df["best_for_skin_types"].str.lower().apply(
        lambda s: any(t in s for t in mentioned)
    )
    filtered = df[mask]
    return filtered if len(filtered) > 0 else df  # fall back if nothing matches

In [25]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()



client = Groq()

SYSTEM_PROMPT = """You are a skincare recommendation assistant. You only answer questions about skincare concerns, ingredients, and product recommendations, using the data provided below. Do not use outside knowledge — if the provided data doesn't clearly answer the question, say you don't have enough information.

Structure every answer as follows:

Top ingredients — List up to 5 ingredients from the provided data most relevant to the user's question, ranked by relevance. For each, give a one-line reason it fits, citing the ingredient name.
Sunscreen follow-up — State clearly whether any of the recommended ingredients require follow-up with sunscreen, based on the data. If none do, say so explicitly rather than omitting the point.
What to avoid — End with a short section naming ingredients or combinations from the data that the user should avoid given their question (e.g., ones flagged as poor combinations with a recommended ingredient, or unsuitable for the stated skin type/concern). If nothing in the data indicates a conflict, say so rather than inventing one.

Cite the ingredient/product name for every claim throughout."""

def generate_response(query, retrieved_df):
    context = "\n".join(
        f"- {row.ingredient_name}: {row.primary_function} (benefits: {row.key_benefits})"
        for _, row in retrieved_df.iterrows()
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"User question: {query}\n\nRelevant data:\n{context}"}
    ]
    completion = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages,
        temperature=0.2,
    )
    return completion.choices[0].message.content

In [26]:
def chatbot_reply(query, top_k=5):
    retrieved, max_sim = retrieve(query, model, collection, df, top_k=top_k)
    if retrieved is None:
        return "This is not my area of expertise — I can help with skincare concerns and product recommendations."
    # filter_by_skin_type was defined above but wasn't actually being called
    # before — wiring it in now that retrieve() returns a larger candidate
    # pool for it to filter down from.
    filtered = filter_by_skin_type(query, retrieved).head(top_k)
    return generate_response(query, filtered)


In [27]:
chatbot_reply("What ingredients help with oily, acne-prone skin?")

'**Top ingredients**  \n1. **Tea Tree Oil** – Natural antimicrobial that spot‑treats blemishes and reduces bacteria, ideal for oily, acne‑prone skin.  \n\n**Sunscreen follow‑up**  \nNone of the recommended ingredients (Tea Tree Oil) are indicated in the data as requiring sunscreen use.  \n\n**What to avoid**  \nNo conflicting ingredients or combinations are listed in the data for oily, acne‑prone skin, so there are no specific avoidances to note.'